# Blood Pressure

This goal of this experiment is to see how 3 factors effect the blood pressure of islanders.

In [18]:
from tqdm.notebook import tqdm
from api.API import IslandsAPI
import pandas as pd
import numpy as np
from time import time_ns,sleep
from math import prod


from api.tasks import * #Load all tasks
from random import shuffle, choice  # Method to randomize groups

tqdm.pandas() #Initialize tqdm


api = IslandsAPI() #Load api
participants = api.get_study_participants() #Get all participants who are contacts
shuffle(participants) #Shuffle the participants list

treatments = {
    "Temperature (C)" : {
        "-20" : SIT_TEMP_NEG20,
        "5" : SIT_TEMP_5,
        "40" : SIT_TEMP_40,
    },
    "Hydration" : {
        "Drink" : DRINK_WATER_250,
        "NoDrink" : None
    },
    "Salty Snack" : {
        "Eaten" : EAT_FRIED_CHIPS,
        "None" : None
    }
}

num_replications = 4
num_treatments = prod([len(group) for name,group in treatments.items()])
num_subsamples = 2

In [111]:
from itertools import product
from random import choices

factors = list(treatments.keys())

levels = [list(treatments[f].items()) for f in factors]

treatment_list = list(product(*levels))

n = num_replications * num_treatments

chosen_people = choices(participants,k=n)

groups = np.array_split(chosen_people, num_treatments)

In [116]:
pd.DataFrame([[person.get_id()] + list((t[0] for t in treatment_list[i]))
 for i, group in enumerate(groups)
 for person in group], columns=(['person_id'] + list(treatments.keys()))).to_csv('blood_pressure_assignments.csv', index=False)

In [123]:
assignments = pd.read_csv('blood_pressure_assignments.csv') #Read csv in

assignments['person_id'] =  assignments['person_id'].map(api.get_person_manager().get_person) # Map all person ids to actual persons

assignments.head() #Display the first 5 columns

,person_id,Temperature (C),Hydration,Salty Snack
0,Person: Id 4xnvnxv57r,-20,Drink,Eaten
1,Person: Id nxuhfejje9,-20,Drink,Eaten
2,Person: Id c4rt6sqnnf,-20,Drink,Eaten
3,Person: Id pabn68x3lg,-20,Drink,Eaten
4,Person: Id e4q92bftqe,-20,Drink,NaN


In [122]:
def wait_with_progress(endtime_ms : float, description: str):#Create a progress bar until a time is complete
    start = time_ns() // 1_000_000
    total_duration = endtime_ms - start

    with tqdm(total=total_duration, unit="ms", desc=description) as pbar:
        last_elapsed = 0

        while (now := time_ns() // 1_000_000) < endtime_ms:
            elapsed = now - start
            if elapsed > last_elapsed:
                pbar.update(elapsed - last_elapsed)
                last_elapsed = elapsed
            sleep(0.1)

        if last_elapsed < total_duration:
            pbar.update(total_duration - last_elapsed)

In [5]:
tqdm.pandas(desc="Starting Blood Pressure Task")
blood_pressure_init = assignments['person_id'].progress_apply(lambda person: person.do_task(MEASURE_BLOOD_PRESSURE)).apply(pd.Series) #Have every person measure their blood glucose levels

wait_with_progress(blood_pressure_init['end_time'].max(), 'Waiting for Blood Pressure Results')

tqdm.pandas(desc='Retrieving Blood Pressure Results')
assignments['person_id'].progress_apply(lambda p: p.update_person())
assignments['base_blood_pressure'] = assignments['person_id'].apply(lambda p: p.get_task_results()[0].result())
assignments.head()

Starting Blood Glucose Task:   0%|          | 0/480 [00:00<?, ?it/s]

Waiting for Blood Glucose Results:   0%|          | 0/12079.0 [00:00<?, ?ms/s]

Retrieving Blood Glucose Results:   0%|          | 0/480 [00:00<?, ?it/s]

,person_id,treatment,base_blood_glucose
0,Person: Id 48tyzy69uc,Control,91 mg/dL
1,Person: Id rvsmw4ntbk,Control,95 mg/dL
2,Person: Id 6xhygzch2q,Control,87 mg/dL
3,Person: Id fkweha6hyt,Control,89 mg/dL
4,Person: Id pd69kskq86,Control,88 mg/dL


In [6]:
def run_task(row, treatment_name):
    return row['person_id'].do_task(treatments[treatment_name][row[treatment_name]])

tqdm.pandas(desc='Starting Sit In Room Task')
waiting_times = assignments.progress_apply(lambda row: run_task(row, 'Temperature (C)'),axis=1).apply(pd.Series)
wait_with_progress(waiting_times['end_time'].max(), "Sitting in environment")



tqdm.pandas(desc="Starting Hydration")
waiting_times = assignments.progress_apply(lambda row: run_task(row, 'Hydration'),axis=1).apply(pd.Series)
wait_with_progress(waiting_times['end_time'].max(), "Hydrating")

tqdm.pandas(desc="Starting Salty Snack")
waiting_times = assignments.progress_apply(lambda row: run_task(row, 'Salty Snack'),axis=1).apply(pd.Series)
wait_with_progress(waiting_times['end_time'].max(), "Eating Snacks")



tqdm.pandas(desc="Measuring Blood Pressure 1")
blood_glucose_results = assignments['person_id'].progress_apply(lambda person: person.do_task(MEASURE_BLOOD_PRESSURE)).apply(pd.Series) #Have every person measure their blood pressure levels
wait_with_progress(blood_glucose_results['end_time'].max(), 'Waiting for Blood Glucose Results 1')


tqdm.pandas(desc="Measuring Blood Pressure 2")
blood_glucose_results = assignments['person_id'].progress_apply(lambda person: person.do_task(MEASURE_BLOOD_PRESSURE)).apply(pd.Series) #Have every person measure their blood pressure levels
wait_with_progress(blood_glucose_results['end_time'].max(), 'Waiting for Blood Glucose Results 2')


tqdm.pandas(desc='Retrieving Blood Pressure Results')
assignments['person_id'].progress_apply(lambda p: p.update_person())


assignments['end_blood_pressure_1'] = assignments['person_id'].apply(lambda p: p.get_task_results()[1].result())

assignments['end_blood_pressure_2'] = assignments['person_id'].apply(lambda p: p.get_task_results()[0].result())

Starting Eat Chocolate Task:   0%|          | 0/480 [00:00<?, ?it/s]

Eating Chocolate:   0%|          | 0/160024.60009765625 [00:00<?, ?ms/s]

Starting Blood Glucose Task:   0%|          | 0/480 [00:00<?, ?it/s]

Waiting for Blood Glucose Results:   0%|          | 0/12340.0 [00:00<?, ?ms/s]

Retrieving Blood Glucose Results:   0%|          | 0/480 [00:00<?, ?it/s]

Data is then saved as a csv: [blood_pressure_results.csv](blood_pressure_results.csv)

In [7]:
assignments['person_id'] = assignments['person_id'].map(lambda x: x.get_id())
assignments['end_blood_pressure_1'] =  assignments['end_blood_pressure_1'].str.split(' ').map(lambda x: x[0]).astype(int) #Clean up rows so blood pressure is an integer

assignments['end_blood_pressure_2'] =  assignments['end_blood_pressure_2'].str.split(' ').map(lambda x: x[0]).astype(int) #Clean up rows so blood pressure is an integer

assignments['base_blood_pressure'] =  assignments['base_blood_pressure'].str.split(' ').map(lambda x: x[0]).astype(int) #Clean up rows so blood pressure is an integer
assignments.to_csv("blood_pressure_results.csv")